In [1]:
"""
FinDER (FinanceRAG) retrieval evaluation harness — modular splitter sweep.
 
Loads FinDER corpus/queries/qrels from LOCAL files (Kaggle export),
tries multiple splitter *types* x multiple (chunk_size, overlap) configs,
encodes each with a SentenceTransformer, retrieves top-k chunks per query,
and scores every (splitter, chunk_size, overlap) combo with Recall@k / NDCG@k.
 
Usage:
    pip install sentence-transformers datasets scikit-learn numpy pandas langchain-text-splitters --break-system-packages
    python finder_splitter_eval.py
"""
 
import os
import math
from dataclasses import dataclass
 
import numpy as np
import pandas as pd
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
 
# ---------------------------------------------------------------------------
# Local file paths (Kaggle export)
# ---------------------------------------------------------------------------

from sklearn.metrics.pairwise import cosine_similarity
from langchain_text_splitters import (
    CharacterTextSplitter,
    RecursiveCharacterTextSplitter
)

CORPUS_PATH = "../finrag_data/finder_corpus.jsonl/corpus.jsonl"
QUERIES_PATH = "../finrag_data/finder_queries.jsonl/queries.jsonl"
QRELS_PATH = "../finrag_data/FinDER_qrels.tsv"
OUT_PATH = "../results/finder_splitter_results.csv"  
QRELS_COLS = ("query_id", "corpus_id", "score")
 
EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"  # or a local path if huggingface.co is unreachable
TOP_K = 10
MAX_QUERIES = 200
MAX_CORPUS_DOCS = None
 
# ---------------------------------------------------------------------------
# Splitter registry — add a new splitter type here, nothing else changes
# ---------------------------------------------------------------------------

D:\conda_envs\finder\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
D:\conda_envs\finder\Lib\site-packages\requests\__init__.py:92: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


In [3]:

def load_local_data():
    corpus_ds = load_dataset("json", data_files=CORPUS_PATH, split="train")
    queries_ds = load_dataset("json", data_files=QUERIES_PATH, split="train")
    try:
        qrels_ds = load_dataset("csv", data_files=QRELS_PATH, delimiter="\t", split="train")
    except Exception as e:
        raise SystemExit(f"Could not load qrels file '{QRELS_PATH}': {e}")
    return corpus_ds, queries_ds, qrels_ds

In [4]:
corpus_ds, queries_ds, qrels_ds = load_local_data()

In [5]:
print( corpus_ds[0:5])

{'_id': ['ADBE20230004', 'ADBE20230006', 'ADBE20230007', 'ADBE20230008', 'ADBE20230010'], 'title': ['ADBE OVERVIEW', 'ADBE OFFERINGS', 'ADBE OFFERINGS', 'ADBE OFFERINGS', 'ADBE OFFERINGS'], 'text': ['Adobe is a global technology company with a mission to change the world through personalized digital experiences. For over four decades, Adobe’s innovations have transformed how individuals, teams, businesses, enterprises, institutions, and governments engage and interact across all types of media. Our products, services and solutions are used around the world to imagine, create, manage, deliver, measure, optimize and engage with content across surfaces and fuel digital experiences. We have a diverse user base that includes consumers, communicators, creative professionals, developers, students, small and medium businesses and enterprises. We are also empowering creators by putting the power of artificial intelligence (“AI”) in their hands, and doing so in ways we believe are responsible. O

In [2]:
# ---------------------------------------------------------------------------
# Splitter registry — add a new splitter type here, nothing else changes
# ---------------------------------------------------------------------------
 
def _char_splitter(chunk_size, overlap):
    def _split(text: str):
        if not text:
            return []
        step = chunk_size - overlap
        out = []
        for start in range(0, len(text), step):
            piece = text[start:start + chunk_size]
            if piece.strip():
                out.append(piece)
            if start + chunk_size >= len(text):
                break
        return out
    return _split
    

In [3]:
def _recursive_char_splitter(chunk_size, overlap):
    from langchain_text_splitters import RecursiveCharacterTextSplitter
    splitter = RecursiveCharacterTextSplitter(chunk_size=chunk_size, chunk_overlap=overlap)
    return splitter.split_text
 
 
def _token_splitter(chunk_size, overlap):
    from langchain_text_splitters import TokenTextSplitter
    splitter = TokenTextSplitter(chunk_size=chunk_size, chunk_overlap=overlap)
    return splitter.split_text
 
 
def _sentence_splitter(chunk_size, overlap):
    from langchain_text_splitters import NLTKTextSplitter
    splitter = NLTKTextSplitter(chunk_size=chunk_size, chunk_overlap=overlap)
    return splitter.split_text
    

In [4]:
SPLITTER_REGISTRY = {
    "char": _char_splitter,
    "recursive_char": _recursive_char_splitter,
    "token": _token_splitter,
    "sentence": _sentence_splitter,
}
 

In [5]:

# ---------------------------------------------------------------------------
# Sweep config — this dict is the single thing you edit to try new combos
# ---------------------------------------------------------------------------
 
SWEEP_CONFIG = {
    "char": [(256, 0), (256, 64), (512, 0), (512, 100), (1024, 0), (1024, 150)],
    "recursive_char": [(256, 64), (512, 100), (1024, 150)],
    "token": [(128, 20), (256, 40)],
    "sentence": [(3, 0), (5, 0)],  # sentence-based splitters interpret chunk_size differently; adjust per splitter as needed
}


In [6]:

@dataclass
class Chunk:
    doc_id: str
    text: str
    

In [7]:

def dcg(rels):
    return sum(r / math.log2(i + 2) for i, r in enumerate(rels))
    

In [8]:

def build_chunks(corpus_rows, splitter_name, chunk_size, overlap):
    splitter_fn = SPLITTER_REGISTRY[splitter_name](chunk_size, overlap)
    chunks = []
    for row in corpus_rows:
        title = (row.get("title") or "").strip()
        body = row.get("text") or ""
        full_text = f"{title}\n{body}" if title else body
        for piece in splitter_fn(full_text):
            if piece and piece.strip():
                chunks.append(Chunk(doc_id=row["_id"], text=piece))
    return chunks
    

In [9]:
def ndcg_at_k(retrieved_doc_ids, relevant_doc_ids, k):
    rels = [1 if d in relevant_doc_ids else 0 for d in retrieved_doc_ids[:k]]
    ideal = sorted(rels, reverse=True)
    ideal_dcg = dcg(ideal)
    return dcg(rels) / ideal_dcg if ideal_dcg > 0 else 0.0
    

In [10]:

def recall_at_k(retrieved_doc_ids, relevant_doc_ids, k):
    if not relevant_doc_ids:
        return 0.0
    hit = len(set(retrieved_doc_ids[:k]) & relevant_doc_ids)
    return hit / len(relevant_doc_ids)
    

In [12]:
def build_relevant_map(qrels_ds, query_ids):
    qcol, dcol, scol = QRELS_COLS
    relevant = {}
    for row in qrels_ds:
        qid = row[qcol]
        if qid not in query_ids:
            continue
        if row.get(scol, 1) and int(row[scol]) > 0:
            relevant.setdefault(qid, set()).add(row[dcol])
    return relevant

In [13]:
def main():
    print(f"Loading local data ...")
    corpus_ds, queries_ds, qrels_ds = load_local_data()
 
    corpus_rows = list(corpus_ds)
    if MAX_CORPUS_DOCS:
        corpus_rows = corpus_rows[:MAX_CORPUS_DOCS]
 
    queries_rows = list(queries_ds)
    if MAX_QUERIES:
        queries_rows = queries_rows[:MAX_QUERIES]
    query_ids = {r["_id"] for r in queries_rows}
 
    relevant = build_relevant_map(qrels_ds, query_ids)
    print(f"Corpus docs: {len(corpus_rows)} | Queries: {len(queries_rows)} | Queries with qrels: {len(relevant)}")

    print(f"Loading embedding model {EMBED_MODEL} ...")
    model = SentenceTransformer(EMBED_MODEL)

    query_texts = [r["text"] for r in queries_rows]
    query_vecs = np.array(model.encode(query_texts, normalize_embeddings=True, show_progress_bar=True))

    results = []
    for splitter_name, configs in SWEEP_CONFIG.items():
        for chunk_size, overlap in configs:
            print(f"\n=== splitter={splitter_name}, chunk_size={chunk_size}, overlap={overlap} ===")
            try:
                chunks = build_chunks(corpus_rows, splitter_name, chunk_size, overlap)
            except ImportError as e:
                print(f"  Skipping {splitter_name}: missing dependency ({e})")
                continue

            chunk_texts = [c.text for c in chunks]
            chunk_doc_ids = [c.doc_id for c in chunks]
            print(f"  {len(chunks)} chunks")

            chunk_vecs = np.array(model.encode(chunk_texts, normalize_embeddings=True,
                                                batch_size=64, show_progress_bar=True))

            scores_matrix = cosine_similarity( query_vecs, chunk_vecs)
            recalls, ndcgs = [], []
            for qi, row in enumerate(queries_rows):
                qid = row["_id"]
                #score = row["score"]
                rel_docs = relevant.get(qid)
                if not rel_docs:
                    continue
                query_scores = scores_matrix[qi]                               # this query's row only
                top_indices = np.argsort(query_scores)[::-1][:TOP_K]  
                retrieved_doc_ids = [chunk_doc_ids[i] for i in top_indices]
                recalls.append(recall_at_k(retrieved_doc_ids, rel_docs, TOP_K))
                ndcgs.append(ndcg_at_k(retrieved_doc_ids, rel_docs, TOP_K))

            results.append({                                      # summarize this splitter config's performance
                    "chunk_size": chunk_size,                          # record the config's chunk size
                    "chunk_overlap": overlap,                    # record the config's overlap
                    "n_chunks": len(chunks),                           # how many chunks this config produced overall
                    "n_queries_scored": len(recalls),                  # how many queries actually contributed to the averages
                    "splitter_name": SPLITTER_REGISTRY[splitter_name],
                    f"recall@{TOP_K}": np.mean(recalls) if recalls else float("nan"),  # mean recall across scored queries
                    f"ndcg@{TOP_K}": np.mean(ndcgs) if ndcgs else float("nan"),        # mean NDCG across scored queries
                }) 
    df = pd.DataFrame(results).sort_values(f"ndcg@{TOP_K}", ascending=False)  # tabulate results, best NDCG first
    print("\n=== Splitter comparison (sorted by NDCG@%d) ===" % TOP_K)  # header before printing the table
    print(df.to_string(index=False))                           # print the full comparison table to stdout
 
    #out_path = "../results/finder_splitter_results.csv"                    # where to persist the results
    df.to_csv(OUT_PATH, index=False)                             # write the comparison table to a CSV file
    print(f"\nSaved results to {OUT_PATH}") 
            

In [14]:
if __name__ == "__main__":
    main()
    

Loading local data ...
Corpus docs: 13867 | Queries: 200 | Queries with qrels: 61
Loading embedding model sentence-transformers/all-MiniLM-L6-v2 ...


Batches: 100%|██████████| 7/7 [00:00<00:00,  7.39it/s]



=== splitter=char, chunk_size=256, overlap=0 ===
  38910 chunks


Batches: 100%|██████████| 608/608 [00:37<00:00, 16.12it/s]



=== splitter=char, chunk_size=256, overlap=64 ===
  45355 chunks


...


=== splitter=token, chunk_size=256, overlap=40 ===
  15610 chunks


Batches: 100%|██████████| 244/244 [00:38<00:00,  6.39it/s]
Created a chunk of size 127, which is longer than the specified 3
Created a chunk of size 185, which is longer than the specified 3
Created a chunk of size 187, which is longer than the specified 3
Created a chunk of size 158, which is longer than the specified 3
Created a chunk of size 151, which is longer than the specified 3
Created a chunk of size 196, which is longer than the specified 3
Created a chunk of size 29, which is longer than the specified 3
Created a chunk of size 292, which is longer than the specified 3
Created a chunk of size 377, which is longer than the specified 3
Created a chunk of size 34, which is longer than the specified 3
Created a chunk of size 248, which is longer than the specified 3
Created a chunk of size 170, which is longer than the specified 3
Created a chunk of size 171, which is longer than the specified 3



=== splitter=sentence, chunk_size=3, overlap=0 ===


Created a chunk of size 223, which is longer than the specified 3
Created a chunk of size 121, which is longer than the specified 3
Created a chunk of size 169, which is longer than the specified 3
Created a chunk of size 246, which is longer than the specified 3
Created a chunk of size 144, which is longer than the specified 3
Created a chunk of size 127, which is longer than the specified 3
Created a chunk of size 252, which is longer than the specified 3
Created a chunk of size 173, which is longer than the specified 3
Created a chunk of size 240, which is longer than the specified 3
Created a chunk of size 283, which is longer than the specified 3
Created a chunk of size 172, which is longer than the specified 3
Created a chunk of size 196, which is longer than the specified 3
Created a chunk of size 290, which is longer than the specified 3
Created a chunk of size 175, which is longer than the specified 3
Created a chunk of size 100, which is longer than the specified 3
Created a 


=== Splitter comparison (sorted by NDCG@10) ===
 chunk_size  chunk_overlap  n_chunks  n_queries_scored  recall@10  ndcg@10
        256             40     15610                61   0.439891 0.341759
       1024            150     16287                61   0.431694 0.336658
       1024              0     16136                61   0.448087 0.333645
       1024            150     17900                61   0.407104 0.333337
        512              0     23266                61   0.464481 0.332685
        512            100     24524                61   0.464481 0.327667
          3              0     37939                61   0.419399 0.321911
          5              0     37939                61   0.419399 0.321911
        128             20     21450                61   0.427596 0.309810
        256             64     45355                61   0.389344 0.302186
        512            100     29996                61   0.390710 0.298703
        256              0     38910               